<a href="https://colab.research.google.com/github/sophia-yang424/pneumonia_class/blob/dev/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
import torchvision.transforms as transforms
from PIL import Image

## Targeted Data Augmentation for Minority Class

To address the class imbalance, we will augment only the minority class (e.g., 'pneumonia' or 'normal', whichever has fewer samples). The goal is to balance the training dataset so that both classes have an equal number of samples. This involves:

1.  **Analyzing the current class distribution** in `train_ds`.
2.  **Separating** the dataset into minority and majority classes.
3.  **Defining augmentation transformations** (e.g., random rotations, flips, brightness adjustments).
4.  **Applying these transformations** repeatedly to the minority class until its count matches the majority class.
5.  **Combining** the augmented minority samples with the original majority samples to form a new, balanced `train_ds`.

This new `train_ds` will then be used in the subsequent preprocessing and training steps.

In [ ]:
!pip install colab-xterm
%load_ext colabxterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 2.9 MB/s eta 0:00:00


In [ ]:
!pip install numpy
!pip install datasets
!pip install transformers
!pip install sklearn
!pip install gradio

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
# --- 1. Load dataset (same one as linear probing exercise) ---
dataset_name = "mmenendezg/pneumonia_x_ray"
ds = load_dataset(dataset_name)

# Subsample for CPU time budget — same scale as the linear probing exercise
train_ds = ds["train"].shuffle(seed=42).select(range(800))
test_ds = ds["test"].shuffle(seed=42).select(range(200))

num_labels = 2  # normal vs pneumonia


README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  110MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.6MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.2MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4187 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1045 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/624 [00:00<?, ? examples/s]

In [ ]:
# 1. Analyze current class distribution
print("Original training dataset class distribution:")
class_counts = train_ds.features["label"].names
label_counts = {label: 0 for label in class_counts}
for item in train_ds:
    label_counts[class_counts[item['label']]] += 1

print(label_counts)

# Identify minority and majority classes
minority_class = min(label_counts, key=label_counts.get)
majority_class = max(label_counts, key=label_counts.get)

minority_label = train_ds.features["label"].str2int(minority_class)
majority_label = train_ds.features["label"].str2int(majority_class)

num_minority = label_counts[minority_class]
num_majority = label_counts[majority_class]

print(f"Minority class: {minority_class} ({num_minority} samples)")
print(f"Majority class: {majority_class} ({num_majority} samples)")

# Calculate how many new samples are needed for the minority class
samples_to_add = num_majority - num_minority
print(f"Samples to add to minority class: {samples_to_add}")

# Filter datasets for minority and majority classes
minority_ds = train_ds.filter(lambda example: example["label"] == minority_label)
majority_ds = train_ds.filter(lambda example: example["label"] == majority_label)

# Define augmentation transformations for the minority class
# These transforms will be applied to PIL Images
augment_transforms = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)), # Assuming input size 224x224 after processor
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(), # Convert to tensor for transforms, will be reconverted to PIL for processor
    transforms.ToPILImage() # Convert back to PIL for the AutoImageProcessor
])

augmented_minority_samples = []
original_minority_images = [item['image'] for item in minority_ds]

# Augment minority class until balanced
for _ in range(samples_to_add):
    # Randomly pick an image from the original minority class to augment
    original_image_index = np.random.randint(0, len(original_minority_images))
    original_image_pil = original_minority_images[original_image_index]

    # Apply augmentation
    augmented_image_pil = augment_transforms(original_image_pil)

    # Create a new sample with the augmented image and original label
    augmented_sample = {
        'image': augmented_image_pil,
        'label': minority_label
    }
    augmented_minority_samples.append(augmented_sample)

# Combine augmented samples with the original minority and majority datasets
# Convert augmented_minority_samples to a Dataset object
from datasets import Dataset
augmented_minority_ds = Dataset.from_list(augmented_minority_samples)

# Concatenate original minority, augmented minority, and original majority
train_ds = Dataset.from_list(minority_ds.to_list() + augmented_minority_ds.to_list() + majority_ds.to_list())

# Shuffle the combined dataset to mix augmented and original samples
train_ds = train_ds.shuffle(seed=42)

# Verify new class distribution
print("\nNew training dataset class distribution after augmentation:")
label_counts_after_aug = {label: 0 for label in class_counts}
for item in train_ds:
    label_counts_after_aug[class_counts[item['label']]] += 1
print(label_counts_after_aug)

Original training dataset class distribution:
{'normal': 200, 'pneumonia': 600}
Minority class: normal (200 samples)
Majority class: pneumonia (600 samples)
Samples to add to minority class: 400


Filter:   0%|          | 0/800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/800 [00:00<?, ? examples/s]


New training dataset class distribution after augmentation:
{'normal': 600, 'pneumonia': 600}


In [ ]:
print(ds["train"].features["label"])

ClassLabel(names=['normal', 'pneumonia'])


In [ ]:
# --- 2. Load processor + model ---
model_name = "microsoft/resnet-18"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)

# No freezing here — this is the fine-tuning exercise, so every parameter
# (backbone + head) keeps its default requires_grad=True and gets updated
# during training.


preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.5k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
import io
from PIL import Image # Ensure PIL.Image is imported if not already

# --- 3. Preprocessing function ---
def preprocess(examples):
    pil_images = []
    for img_data in examples["image"]:
        if isinstance(img_data, Image.Image):
            # If it's already a PIL Image (e.g., from augmented samples)
            pil_images.append(img_data)
        elif isinstance(img_data, dict) and 'bytes' in img_data:
            # If it's a dictionary with 'bytes' (e.g., from original dataset)
            pil_images.append(Image.open(io.BytesIO(img_data['bytes'])))
        else:
            raise TypeError(f"Unexpected image data format: {type(img_data)}")

    inputs = processor([img.convert("RGB") for img in pil_images], return_tensors="pt")
    examples["pixel_values"] = inputs["pixel_values"]
    return examples

train_processed = train_ds.map(preprocess, batched=True)
test_processed = test_ds.map(preprocess, batched=True)


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
# --- 4. Evaluation metric ---
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, preds)
    # Calculate F1 score, specify average='weighted' for multi-class or 'binary' for binary classification
    # Assuming binary classification here, so we'll use 'binary' or 'weighted' if there's imbalance.
    # For binary 'normal' (0) vs 'pneumonia' (1), we can focus on F1 for 'pneumonia' class, or overall weighted.
    # Let's use 'weighted' to account for class imbalance in F1 calculation.
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": accuracy, "f1": f1}


In [ ]:
#5. important: here we init the hyperparams for training
training_args = TrainingArguments(
    output_dir="./fine_tuned_pneumonia_model",
    num_train_epochs=5, # Increased epochs
    per_device_train_batch_size=8,
    learning_rate=1e-6, # Decreased learning rate
    weight_decay=0.1, #increase weight decay to preven tbig weights to prevent overfitting
    eval_strategy="epoch",
    logging_steps=10,
)

In [ ]:
# --- 6. Trainer --- no ensembling, seeds were chosen random and results werent reproducible
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=test_processed,
    compute_metrics=compute_metrics,
)
trainer.train()
# train() wraps these 3: backward, step, zero_grad aka does backprop for however many epochs we specified in traning_args

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Ensembling Models with Different Random Seeds

To perform ensembling, we will train multiple models, each initialized and trained with a different random seed. After training, we will save each model. Then, for inference, we will load all trained models, obtain their predictions (logits) on the test set, and average these logits to get the final ensemble prediction. This often leads to more robust and accurate results.

In [ ]:
import os
import torch
import numpy as np
from transformers import AutoModelForImageClassification, Trainer, TrainingArguments, AutoImageProcessor

#trainign function
def train_and_save_model(seed, model_id, num_labels, processor_instance, training_args_obj, train_ds_processed, test_ds_processed, compute_metrics_fn, base_output_dir):
    # Set seeds for reproducibility for this specific run
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print(f"\n--- Training model with seed: {seed} ---")

    # Re-instantiate the model to ensure fresh weights from pre-trained for each run
    model_for_run = AutoModelForImageClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
    )

    # Create a unique output directory for each model
    current_output_dir = os.path.join(base_output_dir, f"model_seed_{seed}")

    # Ensure the output directory exists
    os.makedirs(current_output_dir, exist_ok=True)

    # Create TrainingArguments for this specific run (can use a copy of the original)
    current_training_args = TrainingArguments(
        output_dir=current_output_dir,
        num_train_epochs=training_args_obj.num_train_epochs,
        per_device_train_batch_size=training_args_obj.per_device_train_batch_size,
        learning_rate=training_args_obj.learning_rate,
        weight_decay=training_args_obj.weight_decay,
        eval_strategy=training_args_obj.eval_strategy,
        logging_steps=training_args_obj.logging_steps,
        load_best_model_at_end=True, # Important for ensembling (use best model, not last)
        metric_for_best_model= "eval_accuracy", # Changed to 'eval_accuracy' for consistency with available metrics
        greater_is_better=True,
        save_strategy="epoch" # Save checkpoints at each epoch
    )

    print(f"  Hyperparameters for this run: LR={current_training_args.learning_rate}, Epochs={current_training_args.num_train_epochs}, Weight Decay={current_training_args.weight_decay}, Batch Size={current_training_args.per_device_train_batch_size}")

    trainer = Trainer(
        model=model_for_run,
        args=current_training_args,
        train_dataset=train_ds_processed,
        eval_dataset=test_ds_processed,
        compute_metrics=compute_metrics_fn,
    )

    trainer.train()
    # Save the final best model (loaded by load_best_model_at_end)
    trainer.save_model(current_output_dir)
    print(f"Model for seed {seed} saved to: {current_output_dir}")
    return current_output_dir # Return the path to the saved model


In [ ]:
#cohort 1: learning rate = 10^-6, epochs = 5
# List of seeds for ensembling
#ensemble_seeds = [42, 101, 202, 303, 404]
ensemble_seeds = [202, 303, 404]
ensemble_model_paths = []
base_ensemble_output_dir = os.path.join(drive_base_path, "fine_tuned_pneumonia_model_ensemble")

# Loop through each seed to train and save a model
for seed in ensemble_seeds:
    model_path = train_and_save_model(
        seed=seed,
        model_id=model_name,
        num_labels=num_labels,
        processor_instance=processor,
        training_args_obj=training_args, # Use the existing training_args object
        train_ds_processed=train_processed,
        test_ds_processed=test_processed,
        compute_metrics_fn=compute_metrics,
        base_output_dir=base_ensemble_output_dir
    )
    ensemble_model_paths.append(model_path)

print(f"\nAll ensemble models trained and saved to: {ensemble_model_paths}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.



--- Training model with seed: 42 ---


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.501397,0.692337,0.545000
2,0.424422,0.560684,0.765000
3,0.361183,0.507106,0.785000
4,0.373090,0.478956,0.820000
5,0.405857,0.465463,0.820000


...

Model for seed 101 saved to: ./fine_tuned_pneumonia_model_ensemble/model_seed_101

--- Training model with seed: 202 ---


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
#just looking at seed 101 but from scratch

#cohort 1: learning rate = 10^-6, epochs = 6
# List of seeds for ensembling
#ensemble_seeds = [42, 101, 202, 303, 404]
ensemble_seeds = [101]
ensemble_model_paths = []
base_ensemble_output_dir = "./fine_tuned_pneumonia_model_ensemble"

training_args = TrainingArguments(
    output_dir="./fine_tuned_pneumonia_model",
    num_train_epochs=6, # Increased epochs
    per_device_train_batch_size=8,
    learning_rate=1e-6, # Decreased learning rate
    weight_decay=0.1, #increase weight decay to preven tbig weights to prevent overfitting
    eval_strategy="epoch",
    logging_steps=10,
)
# Loop through each seed to train and save a model
for seed in ensemble_seeds:
    model_path = train_and_save_model(
        seed=seed,
        model_id=model_name,
        num_labels=num_labels,
        processor_instance=processor,
        training_args_obj=training_args, # Use the existing training_args object
        train_ds_processed=train_processed,
        test_ds_processed=test_processed,
        compute_metrics_fn=compute_metrics,
        base_output_dir=base_ensemble_output_dir
    )
    #ensemble_model_paths.append(model_path)

#print(f"\nAll ensemble models trained and saved to: {ensemble_model_paths}")


--- Training model with seed: 101 ---


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.513992,0.563754,0.755000,0.751226
2,0.394534,0.473591,0.800000,0.797226


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model for seed 101 saved to: ./fine_tuned_pneumonia_model_ensemble/model_seed_101


In [ ]:
import os
import torch
import numpy as np
from transformers import AutoModelForImageClassification, Trainer, TrainingArguments
import json

target_seed = 101
target_model_path = os.path.join(base_ensemble_output_dir, f"model_seed_{target_seed}")

print(f"--- Resuming training for model with seed: {target_seed} for a total of 8 epochs ---")

# Load the already trained model for seed 101
# It's important to load the model from its saved directory to continue its training state
resumed_model = AutoModelForImageClassification.from_pretrained(target_model_path)

# Create new TrainingArguments for this specific resume operation
# Set num_train_epochs to the *new total* epochs (6).
# The Trainer will determine how many more epochs are needed based on existing checkpoints.
resume_training_args = TrainingArguments(
    output_dir=target_model_path, # Output directory is where the existing model is and checkpoints are
    num_train_epochs=8, # New total number of epochs
    per_device_train_batch_size=training_args.per_device_train_batch_size,
    learning_rate=training_args.learning_rate,
    weight_decay=training_args.weight_decay,
    eval_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy", # Changed to 'eval_accuracy' as it is available
    greater_is_better=True,
    save_strategy="epoch"
)

trainer = Trainer(
    model=resumed_model,
    args=resume_training_args,
    train_dataset=train_processed,
    eval_dataset=test_processed,
    compute_metrics=compute_metrics,
)

# Resume training from the last checkpoint found within the output directory.
# The Trainer will automatically detect the completed epochs and continue.
trainer.train(resume_from_checkpoint=True)

# Save the final best model (which load_best_model_at_end will have loaded) after resuming
trainer.save_model(target_model_path)
print(f"Model for seed {target_seed} continued training and saved to: {target_model_path}")

# Ensure the path is in the ensemble_model_paths list and save to disk
if target_model_path not in ensemble_model_paths:
    ensemble_model_paths.append(target_model_path)

ensemble_paths_file = "ensemble_model_paths.json"
with open(ensemble_paths_file, 'w') as f:
    json.dump(ensemble_model_paths, f)
print(f"Updated ensemble model paths saved to: {ensemble_paths_file}")


--- Resuming training for model with seed: 101 for a total of 8 epochs ---


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
6,0.272949,0.372003,0.850000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
#save ensemble paths currently as is to disk and also get f1 for each of the 2 classes per model

In [ ]:
#saving current ensmble methods to disk
import json

# Define the path to save the ensemble model paths
ensemble_paths_file = "ensemble_model_paths.json"

# Save the list to a JSON file
with open(ensemble_paths_file, 'w') as f:
    json.dump(ensemble_model_paths, f)

print(f"Ensemble model paths saved to: {ensemble_paths_file}")

# You can load them back later with:
# with open(ensemble_paths_file, 'r') as f:
#     loaded_ensemble_model_paths = json.load(f)


Ensemble model paths saved to: ensemble_model_paths.json


In [ ]:
import os
import json

# Define the base directory where ensemble models are saved
base_ensemble_output_dir = "./fine_tuned_pneumonia_model_ensemble"
ensemble_paths_file = "ensemble_model_paths.json"

# Reconstruct ensemble_model_paths by finding existing model directories
reconstructed_ensemble_model_paths = []
if os.path.exists(base_ensemble_output_dir):
    for item_name in os.listdir(base_ensemble_output_dir):
        item_path = os.path.join(base_ensemble_output_dir, item_name)
        # Check if it's a directory and looks like a model save directory
        if os.path.isdir(item_path) and item_name.startswith("model_seed_"):
            reconstructed_ensemble_model_paths.append(item_path)

# Sort the paths for consistent ordering
reconstructed_ensemble_model_paths.sort()

# Update the global ensemble_model_paths variable for consistency
ensemble_model_paths = reconstructed_ensemble_model_paths

# Save the reconstructed list to the JSON file
with open(ensemble_paths_file, 'w') as f:
    json.dump(ensemble_model_paths, f)

print(f"Reconstructed and saved ensemble model paths to: {ensemble_paths_file}")
print(f"Current ensemble_model_paths: {ensemble_model_paths}")


Reconstructed and saved ensemble model paths to: ensemble_model_paths.json
Current ensemble_model_paths: ['./fine_tuned_pneumonia_model_ensemble/model_seed_101']


In [ ]:
#now eval the finished ensemble methods, by retrieving them from file on disk and then loading them
#need to run the cell where you defined PredictionDataLoader before running this one, that one was the initial eval cell for right afte ryou train and you stil lhave the ensemble model list defined
from sklearn.metrics import classification_report
import pandas as pd
import json
import torch
from transformers import AutoModelForImageClassification
from tqdm.auto import tqdm
from torch.utils.data import DataLoader # Assuming SimplePredictionDataset and prediction_dataloader are available
import numpy as np

# Define the path to load the ensemble model paths
ensemble_paths_file = "ensemble_model_paths.json"

# Load the ensemble model paths from the JSON file
print(f"Loading ensemble model paths from: {ensemble_paths_file}")
with open(ensemble_paths_file, 'r') as f:
    ensemble_model_paths = json.load(f)

# Re-load all trained models from the paths
ensemble_models = []
for path in ensemble_model_paths:
    print(f"Loading model from {path}...")
    model_loaded = AutoModelForImageClassification.from_pretrained(path) #must load model
    model_loaded.eval() # must then set to evaluation mode bc we arent training these finished ones, alwaayss needed to load any pretrained model whether huggingface or from ur own disk
    ensemble_models.append(model_loaded)

# Assuming `prediction_dataloader`, `true_labels`, and `class_counts_test` are already defined from previous cells.
# If this cell were to be run completely in isolation, those variables would need to be re-initialized.

# Initialize a list to store individual model performance reports
individual_model_f1_reports = []

# Iterate through each model in the ensemble
for model_idx, model_ens in enumerate(ensemble_models):
    print(f"\nEvaluating individual model {model_idx + 1}...")

    model_preds = []
    with torch.no_grad():
        for batch in tqdm(prediction_dataloader, desc=f"Predicting with individual model {model_idx + 1}"):
            pixel_values = batch['pixel_values'].to(model_ens.device)
            outputs = model_ens(pixel_values=pixel_values)
            model_preds.append(np.argmax(outputs.logits.cpu().numpy(), axis=-1))

    model_preds = np.concatenate(model_preds)

    # Generate classification report for the current model
    report_dict = classification_report(true_labels, model_preds, target_names=class_counts_test, output_dict=True)

    # Extract F1 scores for each class
    f1_normal = report_dict['normal']['f1-score']
    f1_pneumonia = report_dict['pneumonia']['f1-score']

    individual_model_f1_reports.append({
        'model_index': model_idx + 1,
        'model_path': ensemble_model_paths[model_idx],
        'f1_normal': f1_normal,
        'f1_pneumonia': f1_pneumonia
    })

    print(f"Model {model_idx + 1} (path: {ensemble_model_paths[model_idx]}) F1 Scores:")
    print(f"  Normal: {f1_normal:.4f}")
    print(f"  Pneumonia: {f1_pneumonia:.4f}")

# Convert the list of reports to a DataFrame for better readability
df_individual_f1_reports = pd.DataFrame(individual_model_f1_reports)
display(df_individual_f1_reports)

Loading ensemble model paths from: ensemble_model_paths.json


FileNotFoundError: [Errno 2] No such file or directory: 'ensemble_model_paths.json'

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 22.5 MB/s eta 0:00:00


### Hyperparameter Search with Optuna

To perform a hyperparameter search, we'll define two key functions:

1.  **`model_init()`**: This function will be called at the beginning of each trial to provide a fresh, untrained model instance. This ensures that each trial starts from the same initial state.
2.  **`hp_space()`**: This function defines the range and type of hyperparameters that Optuna should explore during the search. We'll include `learning_rate`, `num_train_epochs`, and `weight_decay` as primary candidates for optimization.

In [ ]:
import optuna
import os # Import os for path manipulation

# Function to initialize a new model for each trial
def model_init():
    return AutoModelForImageClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
    )

# Define the hyperparameter search space and output directory per trial
def hp_space(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 4e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 3),
        "weight_decay": trial.suggest_float("weight_decay", 0.075, 0.2),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32, 64]), # Expanded batch sizes
        "dropout": trial.suggest_float("dropout", 0.1, 0.5), # Added dropout
        # Dynamically set the output_dir for each trial to Google Drive
        "output_dir": os.path.join(drive_base_path, "hp_search_results", f"trial_{trial.number}"),
    }
    print(f"\n--- Optuna Trial {trial.number} Parameters: {params} ---")
    return params

Now, we'll run the hyperparameter search. This process can take a significant amount of time, as it trains multiple models with different hyperparameter combinations. We will set `n_trials` to a small number (e.g., 5) for demonstration purposes, but for a more robust search, you would typically use a larger number.

The `hyperparameter_search` method will output the best trial found, including the optimal hyperparameters and the best metric achieved.

In [ ]:
import os

# Create a Trainer instance for the hyperparameter search
hp_trainer = Trainer(
    model_init=model_init, # Use the model_init function here
    args=TrainingArguments(
        # The output_dir here will be overridden by the output_dir returned by hp_space for each trial.
        # This ensures each trial's best model is saved in its unique directory.
        output_dir="./hp_search_results", # Default, will be overridden per trial
        per_device_train_batch_size=8, # A default, will be overridden by trial
        learning_rate=1e-5, # A default, will be overridden by trial
        weight_decay=0.01, # A default, will be overridden by trial
        eval_strategy="epoch",
        logging_steps=10,
        load_best_model_at_end=True, # Important: Ensures the best model is saved at the end of each trial
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        save_strategy="epoch", # Save checkpoints at each epoch for best model loading
        report_to="none", # Disable reporting to external services like wandb for simplicity
        seed=42, # Set a fixed seed for reproducible training within each trial
    ),
    train_dataset=train_processed,
    eval_dataset=test_processed,
    compute_metrics=compute_metrics,
)

# Run the hyperparameter search
best_run = hp_trainer.hyperparameter_search(
    direction="maximize", # We want to maximize the metric_for_best_model (accuracy)
    hp_space=hp_space,
    n_trials=50, # Number of trials to run. Increase for a more thorough search.
    backend="optuna",
    # Removed hp_name as it caused a TypeError when internally called as a function
    study_name="pneumonia_hp_search", # Name for the Optuna study for persistent storage
    storage=persistent_db_uri, # Use the persistent URI in Google Drive
    load_if_exists=True # Instruct Optuna to load the study if it already exists
)

print("Best hyperparameters found for the overall study:")
print(best_run)
print("\nEach trial's best model (and its checkpoints) will be saved in a subfolder within './hp_search_results' (e.g., './hp_search_results/trial_0', './hp_search_results/trial_1').")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[I 2026-08-29 19:52:42,634] Using an existing study with name 'pneumonia_hp_search' instead of creating a new one.
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] Trying to set dropout in the hyperparameter search but there is no corresponding field in `TrainingArguments`.


...

[I 2026-08-29 20:42:17,933] Trial 20 finished with value: 1.8078952897683451 and parameters: {'learning_rate': 3.656262595507485e-05, 'num_train_epochs': 2, 'weight_decay': 0.16314418855433685, 'per_device_train_batch_size': 16, 'dropout': 0.43737717388149894}. Best is trial 18 with value: 1.8181663837011885.
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] Trying to set dropout in the hyperparameter search but there is no corresponding field in `TrainingArguments`.



--- Optuna Trial 21 Parameters: {'learning_rate': 3.9660464433688315e-05, 'num_train_epochs': 2, 'weight_decay': 0.17293165250079687, 'per_device_train_batch_size': 16, 'dropout': 0.4145649304834811, 'output_dir': '/content/drive/MyDrive/my_ml_models/hp_search_results/trial_21'} ---


Epoch,Training Loss,Validation Loss


### Making `optuna_study.db` persistent

Since files in the Colab runtime are temporary, we need to move `optuna_study.db` to Google Drive to ensure it persists across sessions. We will also update the `storage` path in the `hp_trainer.hyperparameter_search` function to point to this new location in Drive.

In [ ]:
import os
import shutil

# Define the target directory in Google Drive for the Optuna DB
optuna_db_drive_path = os.path.join(drive_base_path, "optuna_db")
os.makedirs(optuna_db_drive_path, exist_ok=True)

source_db_path = "./optuna_study.db"
destination_db_path = os.path.join(optuna_db_drive_path, "optuna_study.db")

if os.path.exists(source_db_path):
    print(f"Moving {source_db_path} to {destination_db_path}...")
    shutil.move(source_db_path, destination_db_path)
    print("Move successful.")
else:
    print(f"'{source_db_path}' not found. Assuming it either doesn't exist yet or has already been moved.")

# Update the path to be used by Optuna
persistent_db_uri = f"sqlite:///{destination_db_path}"
print(f"Persistent Optuna DB URI: {persistent_db_uri}")

Moving ./optuna_study.db to /content/drive/MyDrive/my_ml_models/optuna_db/optuna_study.db...
Move successful.
Persistent Optuna DB URI: sqlite:////content/drive/MyDrive/my_ml_models/optuna_db/optuna_study.db


In [ ]:
import os
import json
import optuna
from optuna.trial import TrialState, create_trial

def backfill_optuna_study(drive_base_path, study_name, storage_uri):
    hp_results_path = os.path.join(drive_base_path, "hp_search_results")

    if not os.path.exists(hp_results_path):
        print(f"No results found at {hp_results_path} to backfill.")
        return

    # Load the study
    study = optuna.create_study(study_name=study_name, storage=storage_uri, load_if_exists=True, direction="maximize")

    # Get all trial folders
    all_items = os.listdir(hp_results_path)
    trial_folders = [d for d in all_items if d.startswith("trial_") and os.path.isdir(os.path.join(hp_results_path, d))]

    print(f"Total trial folders found in Drive: {len(trial_folders)}")

    # Track existing trial numbers
    existing_trial_numbers = {t.number for t in study.trials}

    for folder in sorted(trial_folders, key=lambda x: int(x.split('_')[1])):
        try:
            trial_num = int(folder.split('_')[1])
        except ValueError:
            continue

        if trial_num in existing_trial_numbers:
            continue

        folder_path = os.path.join(hp_results_path, folder)
        state_file = os.path.join(folder_path, "trainer_state.json")
        results_file = os.path.join(folder_path, "all_results.json")

        # If the trial completed successfully, sync as COMPLETE
        if os.path.exists(state_file) and os.path.exists(results_file):
            try:
                with open(state_file, 'r') as f: state_data = json.load(f)
                with open(results_file, 'r') as f: res_data = json.load(f)

                value = state_data.get("best_metric")
                params = {
                    "learning_rate": res_data.get("learning_rate"),
                    "num_train_epochs": int(res_data.get("epoch", 0)) if res_data.get("epoch") else None,
                    "weight_decay": res_data.get("weight_decay"),
                    "per_device_train_batch_size": res_data.get("train_batch_size"),
                    "dropout": res_data.get("dropout")
                }
                params = {k: v for k, v in params.items() if v is not None}

                trial = create_trial(
                    state=TrialState.COMPLETE,
                    value=value,
                    params=params,
                    distributions={
                        "learning_rate": optuna.distributions.FloatDistribution(1e-6, 4e-5, log=True),
                        "num_train_epochs": optuna.distributions.IntDistribution(2, 3),
                        "weight_decay": optuna.distributions.FloatDistribution(0.075, 0.2),
                        "per_device_train_batch_size": optuna.distributions.CategoricalDistribution([8, 16, 32, 64]),
                        "dropout": optuna.distributions.FloatDistribution(0.1, 0.5)
                    }
                )
                study.add_trial(trial)
                print(f"Synced Trial {trial_num} as COMPLETE (Score: {value:.4f})")
            except Exception as e:
                print(f"Error parsing results for Trial {trial_num}: {e}")
        else:
            # If result files are missing, sync as FAIL so the ID is 'used' in the DB
            trial = create_trial(
                state=TrialState.FAIL,
                params={},
                distributions={}
            )
            study.add_trial(trial)
            print(f"Synced Trial {trial_num} as FAIL (Missing result files). DB ID is now reserved.")

    print(f"\nFinal check: Study '{study_name}' now has {len(study.trials)} trials in the database.")

# Run the backfill
backfill_optuna_study(drive_base_path, "pneumonia_hp_search", persistent_db_uri)

[I 2026-08-29 19:51:33,622] Using an existing study with name 'pneumonia_hp_search' instead of creating a new one.


Total trial folders found in Drive: 10
Synced Trial 4 as FAIL (Missing result files). DB ID is now reserved.
Synced Trial 5 as FAIL (Missing result files). DB ID is now reserved.
Synced Trial 6 as FAIL (Missing result files). DB ID is now reserved.
Synced Trial 7 as FAIL (Missing result files). DB ID is now reserved.
Synced Trial 8 as FAIL (Missing result files). DB ID is now reserved.
Synced Trial 9 as FAIL (Missing result files). DB ID is now reserved.

Final check: Study 'pneumonia_hp_search' now has 10 trials in the database.


Now that the database has been moved to Google Drive (or its target directory created), the next step is to modify the `hp_trainer.hyperparameter_search` call to use this persistent path for the `storage` parameter. I'll modify cell `ff06c8f4` to reflect this change.

### 🚀 Saving Models to Google Drive

To persist your models beyond the Colab session, you need to save them to your Google Drive. First, run the cell below to mount your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define a base path in your Google Drive for saving models
# You can change 'my_ml_models' to any folder name you prefer in your Drive
drive_base_path = '/content/drive/MyDrive/my_ml_models'

# Create the directory if it doesn't exist
import os
os.makedirs(drive_base_path, exist_ok=True)
print(f"Google Drive mounted. Models will be saved to: {drive_base_path}")

Mounted at /content/drive
Google Drive mounted. Models will be saved to: /content/drive/MyDrive/my_ml_models


Now that your Drive is mounted, you would modify the `output_dir` paths in your training and hyperparameter search configurations to point to a location within `/content/drive/MyDrive/`.

For example, if you wanted to save the results of your hyperparameter search to Drive, you would modify the `hp_space` function (cell `315be669`) to use a Drive path:

```python
# In hp_space(trial) function:
# output_dir = os.path.join(drive_base_path, "hp_search_results", f"trial_{trial.number}")
```

And similarly for your ensemble models (e.g., in cell `bbc5fe35`):

```python
# For ensemble models:
# base_ensemble_output_dir = os.path.join(drive_base_path, "fine_tuned_pneumonia_model_ensemble")
```

Would you like me to update the relevant cells to save to Google Drive?

### Accessing the Optuna SQLite Database

In [ ]:
import optuna
import os

# The database file should be in the current working directory
# Change this to use the persistent_db_uri defined previously
db_path = persistent_db_uri # Use the URI pointing to Google Drive

# Load the study from the database
# Ensure the study_name matches what was used during the hyperparameter_search
try:
    study = optuna.load_study(study_name="pneumonia_hp_search", storage=db_path)
    print(f"Successfully loaded study: {study.study_name}")
    print(f"Number of trials: {len(study.trials)}")

    # Print details for each trial
    for i, trial in enumerate(study.trials):
        print(f"\nTrial {i+1} (State: {trial.state}):")
        print(f"  Value: {trial.value}")
        print(f"  Parameters: {trial.params}")
        print(f"  User Attributes: {trial.user_attrs}")
        print(f"  System Attributes: {trial.system_attrs}")
except Exception as e:
    print(f"Error loading Optuna study: {e}")
    print(f"Please ensure the database at '{db_path}' exists and the study_name 'pneumonia_hp_search' is correct.")

# Optional: display the study dashboard (requires Optuna-Dashboard to be installed and run separately)
# !pip install optuna-dashboard
# !optuna-dashboard sqlite:///optuna_study.db

Successfully loaded study: pneumonia_hp_search
Number of trials: 4

Trial 1 (State: TrialState.COMPLETE):
  Value: 1.6748743393377197
  Parameters: {'learning_rate': 2.9636570208922116e-06, 'num_train_epochs': 3, 'weight_decay': 0.1484686840009943, 'per_device_train_batch_size': 8, 'dropout': 0.4402767043515362}
  User Attributes: {}
  System Attributes: {}

Trial 2 (State: TrialState.COMPLETE):
  Value: 1.2817307692307693
  Parameters: {'learning_rate': 1.6202097141051437e-06, 'num_train_epochs': 2, 'weight_decay': 0.14229104087583289, 'per_device_train_batch_size': 16, 'dropout': 0.16105411233739494}
  User Attributes: {}
  System Attributes: {}

Trial 3 (State: TrialState.FAIL):
  Value: None
  Parameters: {'learning_rate': 5.340914916044551e-06, 'num_train_epochs': 3, 'weight_decay': 0.16100288219839026, 'per_device_train_batch_size': 64, 'dropout': 0.37170981470880815}
  User Attributes: {}
  System Attributes: {}

Trial 4 (State: TrialState.FAIL):
  Value: None
  Parameters: {'le

In [ ]:
best_run_hyperparameters = {'learning_rate': 3.713408826464528e-05, 'num_train_epochs': 3, 'weight_decay': 0.09730862815310555, 'per_device_train_batch_size': 16}
print(f"Best trial parameters: {best_run_hyperparameters}")


# Extract best hyperparameters
best_lr = best_run_hyperparameters['learning_rate']
best_epochs = best_run_hyperparameters['num_train_epochs']
best_weight_decay = best_run_hyperparameters['weight_decay']
best_batch_size = best_run_hyperparameters['per_device_train_batch_size']

print(f"\nUsing best hyperparameters: LR={best_lr}, Epochs={best_epochs}, Weight Decay={best_weight_decay}, Batch Size={best_batch_size}")

# Define new training arguments using the best hyperparameters
best_training_args = TrainingArguments(
    output_dir=os.path.join(drive_base_path, "fine_tuned_pneumonia_model_best_hp"), # Unique output directory for best HP models
    num_train_epochs=best_epochs,
    per_device_train_batch_size=best_batch_size,
    learning_rate=best_lr,
    weight_decay=best_weight_decay,
    eval_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    save_strategy="epoch",
    report_to="none",
    seed=42, # Set a fixed seed for reproducible training within each model trained
)

# Define seeds for training multiple models with the best hyperparameters
ensemble_seeds_for_best_hp = [42, 101, 202, 303, 404] # Example seeds, you can modify this list
ensemble_model_paths_best_hp = []
base_output_dir_best_hp = os.path.join(drive_base_path, "fine_tuned_pneumonia_model_best_hp_ensemble")

print("\n--- Training multiple models with best hyperparameters and different random seeds ---")
for seed in ensemble_seeds_for_best_hp:
    model_path = train_and_save_model(
        seed=seed,
        model_id=model_name,
        num_labels=num_labels,
        processor_instance=processor,
        training_args_obj=best_training_args, # Use the best_training_args here
        train_ds_processed=train_processed,
        test_ds_processed=test_processed,
        compute_metrics_fn=compute_metrics,
        base_output_dir=base_output_dir_best_hp # Save under a dedicated folder in Drive
    )
    ensemble_model_paths_best_hp.append(model_path)

print(f"\nAll models with best hyperparameters trained and saved to: {ensemble_model_paths_best_hp}")

Best trial parameters: {'learning_rate': 3.713408826464528e-05, 'num_train_epochs': 3, 'weight_decay': 0.09730862815310555, 'per_device_train_batch_size': 16}

Using best hyperparameters: LR=3.713408826464528e-05, Epochs=3, Weight Decay=0.09730862815310555, Batch Size=16

--- Training multiple models with best hyperparameters and different random seeds ---

--- Training model with seed: 42 ---


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Hyperparameters for this run: LR=3.713408826464528e-05, Epochs=3, Weight Decay=0.09730862815310555, Batch Size=16


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model for seed 404 saved to: /content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_404

All models with best hyperparameters trained and saved to: ['/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_42', '/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_101', '/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_202', '/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_303', '/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_404']


In [ ]:
#best one so far was seed 42 for the trial 7 hyperparams, lets eval them here
import os
import torch
import numpy as np
from transformers import AutoModelForImageClassification
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

# 1. Define the specific checkpoint path provided by the user
target_checkpoint_path = '/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_42/checkpoint-150'

# 2. Wrapper for evaluation dataset (ensure test_processed is available from cell de513597)
class EvalDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            'pixel_values': torch.tensor(item['pixel_values']),
            'labels': torch.tensor(item['label'])
        }

# 3. Load the model and set to eval mode
print(f"Loading model from {target_checkpoint_path}...")
specific_model = AutoModelForImageClassification.from_pretrained(target_checkpoint_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
specific_model.to(device)
specific_model.eval()

# 4. Prepare DataLoader
eval_loader = DataLoader(EvalDataset(test_processed), batch_size=16)

# 5. Run Inference
preds = []
true_labels = []

with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating Checkpoint-150"):
        inputs = batch['pixel_values'].to(device)
        outputs = specific_model(pixel_values=inputs)
        preds.append(np.argmax(outputs.logits.cpu().numpy(), axis=-1))
        true_labels.append(batch['labels'].numpy())

# 6. Generate Report
preds = np.concatenate(preds)
true_labels = np.concatenate(true_labels)
target_names = ds['train'].features['label'].names

print(f"\n--- Classification Report for: {os.path.basename(target_checkpoint_path)} ---")
print(classification_report(true_labels, preds, target_names=target_names))

Loading model from /content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_42/checkpoint-150...


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

Evaluating Checkpoint-150:   0%|          | 0/13 [00:00<?, ?it/s]


--- Classification Report for: checkpoint-150 ---
              precision    recall  f1-score   support

      normal       0.99      0.85      0.91        84
   pneumonia       0.90      0.99      0.94       116

    accuracy                           0.93       200
   macro avg       0.94      0.92      0.93       200
weighted avg       0.94      0.93      0.93       200



In [ ]:
import os

# Create a Trainer instance for the hyperparameter search
hp_trainer = Trainer(
    model_init=model_init, # Use the model_init function here
    args=TrainingArguments(
        output_dir="./hp_search_results", # Output directory for HP search
        # These are default args; actual trial args will be set by hp_space
        per_device_train_batch_size=8, # A default, will be overridden by trial
        learning_rate=1e-5, # A default, will be overridden by trial
        weight_decay=0.01, # A default, will be overridden by trial
        eval_strategy="epoch",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        save_strategy="epoch", # Save checkpoints at each epoch for best model loading
        report_to="none", # Disable reporting to external services like wandb for simplicity
    ),
    train_dataset=train_processed,
    eval_dataset=test_processed,
    compute_metrics=compute_metrics,
)

# Run the hyperparameter search
best_run = hp_trainer.hyperparameter_search(
    direction="maximize", # We want to maximize the metric_for_best_model (accuracy)
    hp_space=hp_space,
    n_trials=5, # Number of trials to run. Increase for a more thorough search.
    backend="optuna",
    # Optionally, specify a study name for Optuna dashboard
    # study_name="pneumonia_hp_search",
    # Also, you can specify storage for persistent study results (e.g., "sqlite:///db.sqlite3")
)

print("Best hyperparameters found:")
print(best_run)

# You can also access the best trial's metrics directly
# print(f"Best trial's accuracy: {best_run.metrics['eval_accuracy']}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[I 2026-08-28 02:59:54,971] A new study created in memory with name: no-name-4a11a25b-b959-4552-b165-fac55149f90d
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.



--- Optuna Trial 0 Parameters: {'learning_rate': 2.607230403082739e-05, 'num_train_epochs': 14, 'weight_decay': 0.2053050238084639, 'per_device_train_batch_size': 32} ---


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[W 2026-08-28 03:00:05,194] Trial 0 failed with parameters: {'learning_rate': 2.607230403082739e-05, 'num_train_epochs': 14, 'weight_decay': 0.2053050238084639, 'per_device_train_batch_size': 32} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/usr/local/lib/python3.13/dist-packages/transformers/integrations/integration_utils.py", line 257, in _objective
    trainer.train(resume_from_checkpoint=checkpoint, trial=trial)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1449, in train
    return i

KeyboardInterrupt: 

### Median Pruning Configuration
We use the `MedianPruner` to stop unpromising trials if their intermediate metrics are below the median of previous trials at the same step.

In [ ]:
import optuna

# Initialize the MedianPruner
# n_startup_trials: Pruning starts only after 5 trials are completed
# n_warmup_steps: Trials are not pruned before the 1st epoch/report
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=1
)

# To apply this to your study, use:
# study = optuna.create_study(
#     study_name="pneumonia_hp_search",
#     storage=persistent_db_uri,
#     load_if_exists=True,
#     direction="maximize",
#     pruner=pruner
# )

print("MedianPruner initialized. Remember to pass 'pruner=pruner' to create_study/load_study.")

In [ ]:
import os

# Create a Trainer instance for the hyperparameter search
hp_trainer = Trainer(
    model_init=model_init, # Use the model_init function here
    args=TrainingArguments(
        output_dir="./hp_search_results", # Output directory for HP search
        # These are default args; actual trial args will be set by hp_space
        per_device_train_batch_size=8, # A default, will be overridden by trial
        learning_rate=1e-5, # A default, will be overridden by trial
        weight_decay=0.01, # A default, will be overridden by trial
        eval_strategy="epoch",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        save_strategy="epoch", # Save checkpoints at each epoch for best model loading
        report_to="none", # Disable reporting to external services like wandb for simplicity
    ),
    train_dataset=train_processed,
    eval_dataset=test_processed,
    compute_metrics=compute_metrics,
)

print("Re-running hyperparameter search...")
# Run the hyperparameter search
best_run = hp_trainer.hyperparameter_search(
    direction="maximize", # We want to maximize the metric_for_best_model (accuracy)
    hp_space=hp_space,
    n_trials=5, # Number of trials to run. Increase for a more thorough search.
    backend="optuna",
    # Optionally, specify a study name for Optuna dashboard
    # study_name="pneumonia_hp_search",
    # Also, you can specify storage for persistent study results (e.g., "sqlite:///db.sqlite3")
)

print("Best hyperparameters found:")
print(best_run)

# You can also access the best trial's metrics directly
# print(f"Best trial's accuracy: {best_run.metrics['eval_accuracy']}")

In [ ]:
print(best_run)

NameError: name 'best_run' is not defined

In [ ]:
#cohort 2
new_learning_rate = 1e-5
new_num_train_epochs = 3

# Create new TrainingArguments for the second cohort
training_args_cohort2 = TrainingArguments(
    output_dir=f"./fine_tuned_pneumonia_model_ensemble_lr_{new_learning_rate}_epochs_{new_num_train_epochs}",
    num_train_epochs=new_num_train_epochs,
    per_device_train_batch_size=training_args.per_device_train_batch_size,
    learning_rate=new_learning_rate,
    weight_decay=training_args.weight_decay,
    eval_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy", # Changed to 'eval_accuracy' as it is available
    greater_is_better=True,
    save_strategy="epoch"
)

print(f"\n--- Training second ensemble cohort (LR={new_learning_rate}, Epochs={new_num_train_epochs}) ---")

# Loop through each seed to train and save models for the second cohort
for seed in ensemble_seeds:
    model_path = train_and_save_model(
        seed=seed,
        model_id=model_name,
        num_labels=num_labels,
        processor_instance=processor,
        training_args_obj=training_args_cohort2, # Use the new training_args object
        train_ds_processed=train_processed,
        test_ds_processed=test_processed,
        compute_metrics_fn=compute_metrics,
        base_output_dir=training_args_cohort2.output_dir # Use the output_dir from new training_args
    )
    ensemble_model_paths.append(model_path)

print(f"\nAll ensemble models (including second cohort) trained and saved to: {ensemble_model_paths}")


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.



--- Training second ensemble cohort (LR=1e-05, Epochs=3) ---

--- Training model with seed: 42 ---


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.202060,0.314349,0.860000
2,0.122491,0.308732,0.875000
3,0.091880,0.330011,0.875000


...

Model for seed 42 saved to: ./fine_tuned_pneumonia_model_ensemble_lr_1e-05_epochs_3/model_seed_42

--- Training model with seed: 101 ---


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Performing Ensemble Prediction

Now that we have multiple models trained with different seeds, we will load them and use them to predict on the test dataset. The final ensemble prediction will be derived by averaging the raw logits from each model, and then taking the argmax to get the final class.

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForImageClassification, AutoImageProcessor
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report

# Load all trained models
ensemble_models = []
for path in ensemble_model_paths:
    print(f"Loading model from {path}...")
    model_loaded = AutoModelForImageClassification.from_pretrained(path)
    model_loaded.eval() # Set to evaluation mode
    ensemble_models.append(model_loaded)

# Prepare test data for prediction
# Create a DataLoader for test_processed to get batches
# We need to map 'pixel_values' and 'labels' to 'input_ids' and 'labels' if Trainer expects them that way
# However, for direct model inference, we just need 'pixel_values'

# Assuming test_processed has 'pixel_values' and 'label' (for true labels)
# Create a simple dataset wrapper for the DataLoader
class SimplePredictionDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            'pixel_values': torch.tensor(item['pixel_values']), # Ensure it's a tensor
            'labels': torch.tensor(item['label']) # For calculating metrics
        }

prediction_dataset = SimplePredictionDataset(test_processed)
prediction_dataloader = DataLoader(prediction_dataset, batch_size=training_args.per_device_eval_batch_size)

all_logits = []
true_labels = []

# Get predictions from each model
for model_idx, model_ens in enumerate(ensemble_models):
    print(f"Getting predictions from ensemble model {model_idx + 1}...")
    model_logits = []
    with torch.no_grad():
        for batch in tqdm(prediction_dataloader, desc=f"Predicting with model {model_idx + 1}"):
            pixel_values = batch['pixel_values'].to(model_ens.device)
            outputs = model_ens(pixel_values=pixel_values)
            model_logits.append(outputs.logits.cpu().numpy())
            if model_idx == 0: # Only collect true labels once
                true_labels.append(batch['labels'].cpu().numpy())
    all_logits.append(np.concatenate(model_logits))

true_labels = np.concatenate(true_labels)



Loading model from ./fine_tuned_pneumonia_model_ensemble/model_seed_101...


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

Getting predictions from ensemble model 1...


Predicting with model 1:   0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
!ls -F

ensemble_model_paths.json  fine_tuned_pneumonia_model_ensemble/  sample_data/


In [ ]:
# Per-model classification reports (before averaging)
for model_idx, logits in enumerate(all_logits):
    model_preds = np.argmax(logits, axis=-1)
    print(f"\n--- Classification Report: Model {model_idx + 1} ---")
    print(classification_report(true_labels, model_preds, target_names=class_counts_test))

# Average the logits across all models
ensemble_averaged_logits = np.mean(all_logits, axis=0)

# Get final predictions from averaged logits
ensemble_preds = np.argmax(ensemble_averaged_logits, axis=-1)

# Evaluate ensemble performance
ensemble_accuracy = accuracy_score(true_labels, ensemble_preds)
print(f"\nEnsemble Accuracy: {ensemble_accuracy:.4f}")

# Generate a classification report for detailed metrics
report = classification_report(true_labels, ensemble_preds, target_names=class_counts_test)
print("\nEnsemble Classification Report:")
print(report)


--- Classification Report: Model 1 ---
              precision    recall  f1-score   support

      normal       0.77      0.81      0.79        84
   pneumonia       0.86      0.83      0.84       116

    accuracy                           0.82       200
   macro avg       0.81      0.82      0.82       200
weighted avg       0.82      0.82      0.82       200


--- Classification Report: Model 2 ---
              precision    recall  f1-score   support

      normal       0.85      0.75      0.80        84
   pneumonia       0.83      0.91      0.87       116

    accuracy                           0.84       200
   macro avg       0.84      0.83      0.83       200
weighted avg       0.84      0.84      0.84       200


Ensemble Accuracy: 0.8500

Ensemble Classification Report:
              precision    recall  f1-score   support

      normal       0.86      0.77      0.81        84
   pneumonia       0.85      0.91      0.88       116

    accuracy                           0.8

### Detailed Evaluation of a Specific Hyperparameter Search Model

To evaluate a specific model from the hyperparameter search, we need to load it from its saved directory in Google Drive and then run predictions on the test set. We'll use the `classification_report` to get detailed metrics for both classes.

In [ ]:
#eval for a specific saved model from hyperparam search we did
import os
from transformers import AutoModelForImageClassification
import torch
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import json # Import json for loading trainer_state.json
from torch.utils.data import DataLoader # Import DataLoader

# Assuming `test_processed` is defined from previous cells
# If this cell were to be run completely in isolation, `test_processed` would need to be re-initialized.

# Define SimplePredictionDataset (copied from a9769c63)
class SimplePredictionDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            'pixel_values': torch.tensor(item['pixel_values']), # Ensure it's a tensor
            'labels': torch.tensor(item['label']) # For calculating metrics from original dataset
        }

# Prepare test data for prediction
prediction_dataset = SimplePredictionDataset(test_processed)
# Assuming `training_args` is defined globally or has a default `per_device_eval_batch_size`
# If `training_args` is not defined, you might need to set a default batch size directly here, e.g., `batch_size=32`
prediction_dataloader = DataLoader(prediction_dataset, batch_size=training_args.per_device_eval_batch_size)

# Get true labels
true_labels = []
for batch in prediction_dataloader:
    true_labels.append(batch['labels'].cpu().numpy())
true_labels = np.concatenate(true_labels)

# Class names for the binary labels (0: normal, 1: pneumonia)
# Assuming `class_counts_test` is available from original data loading
class_counts_test = ds["train"].features["label"].names # Re-fetch class names from original dataset if needed

# Specify the trial number you want to evaluate
target_trial_number = 7 # Example: evaluating trial_2

# Construct the path to the specific trial's root directory where the Trainer's output_dir was set
trial_output_dir = os.path.join(drive_base_path, "hp_search_results", f"trial_{target_trial_number}")

specific_model_load_path = None

print(f"Attempting to find and load model for Trial {target_trial_number} from: {trial_output_dir}")

try:
    # Strategy 1: Check if the model (config.json) was saved directly in the trial_output_dir
    if os.path.exists(os.path.join(trial_output_dir, "config.json")):
        specific_model_load_path = trial_output_dir
        print(f"Found model config.json directly in trial output directory: {specific_model_load_path}")
    else:
        # Strategy 2: Check for trainer_state.json within trial_output_dir to get best_model_checkpoint
        trainer_state_file = os.path.join(trial_output_dir, "trainer_state.json")
        if os.path.exists(trainer_state_file):
            print(f"Found trainer_state.json in trial output directory: {trainer_state_file}. Extracting best_model_checkpoint...")
            with open(trainer_state_file, 'r') as f:
                trainer_state = json.load(f)
            if "best_model_checkpoint" in trainer_state and trainer_state["best_model_checkpoint"] is not None:
                # Use the absolute path provided in trainer_state.json
                specific_model_load_path = trainer_state["best_model_checkpoint"]
                print(f"Using best_model_checkpoint path from trainer_state.json: {specific_model_load_path}")
            else:
                print("trainer_state.json found, but 'best_model_checkpoint' key is missing or None. Searching for valid checkpoint subdirectory.")
        else:
            print(f"trainer_state.json not found in trial output directory ({trial_output_dir}). Searching for valid checkpoint subdirectory.")

        # Strategy 3 (Fallback): Search for valid checkpoint subdirectories if specific_model_load_path is still None
        if specific_model_load_path is None:
            candidate_paths = []
            for root, dirs, files in os.walk(trial_output_dir):
                for d in dirs:
                    if d.startswith("checkpoint-"):
                        candidate_paths.append(os.path.join(root, d))

            if candidate_paths:
                # Sort candidates by checkpoint number (heuristic for 'latest' or 'best' if no other info)
                sorted_candidates = sorted(candidate_paths, key=lambda x: int(os.path.basename(x).split('-')[1]) if os.path.basename(x).startswith('checkpoint-') else 0, reverse=True)
                specific_model_load_path = sorted_candidates[0]
                print(f"Falling back to latest checkpoint directory found: {specific_model_load_path}")
            else:
                print(f"No valid model (config.json) or checkpoint directory found within {trial_output_dir}. Cannot load model.")
                raise FileNotFoundError(f"No model found at {trial_output_dir} or any checkpoint therein.")

    # Final check: ensure the determined path actually contains a config.json
    if specific_model_load_path is None or not os.path.exists(os.path.join(specific_model_load_path, "config.json")):
        raise FileNotFoundError(f"The determined model load path '{specific_model_load_path}' does not contain 'config.json'. Model not found or incomplete.")

    # Load the specific model from the determined path
    specific_model = AutoModelForImageClassification.from_pretrained(specific_model_load_path)
    specific_model.eval() # Set to evaluation mode

    specific_model_preds = []
    with torch.no_grad():
        for batch in tqdm(prediction_dataloader, desc=f"Predicting with Trial {target_trial_number} Model"):
            pixel_values = batch['pixel_values'].to(specific_model.device)
            outputs = specific_model(pixel_values=pixel_values)
            specific_model_preds.append(np.argmax(outputs.logits.cpu().numpy(), axis=-1))

    specific_model_preds = np.concatenate(specific_model_preds)

    print(f"\n--- Detailed Classification Report for Trial {target_trial_number} Model ---")
    print(classification_report(true_labels, specific_model_preds, target_names=class_counts_test))

except Exception as e:
    error_path = specific_model_load_path if specific_model_load_path else trial_output_dir
    print(f"Error evaluating model from {error_path}: {e}")
    print("Please ensure the path is correct and the model was successfully saved.")

NameError: name 'test_processed' is not defined

/tmp/ipykernel_1162/2516009190.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=42))


In [ ]:
# Check class distribution in the test dataset
print("Test dataset class distribution:")
class_counts_test = test_ds.features["label"].names
label_counts_test = {label: 0 for label in class_counts_test}
for item in test_ds:
    label_counts_test[class_counts_test[item['label']]] += 1
print(label_counts_test)


Test dataset class distribution:
{'normal': 84, 'pneumonia': 116}


In [ ]:
# --- 7. Evaluate ---
results = trainer.evaluate()
print(results)

# --- 8. Save fine-tuned model ---
trainer.save_model("./fine_tuned_pneumonia_model")

In [ ]:
import gradio as gr
import torch
import os
from transformers import AutoModelForImageClassification, AutoImageProcessor

# Define the paths to check
base_path = "/content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_42"
checkpoint_path = os.path.join(base_path, "checkpoint-150")

# Determine which path has the config.json
if os.path.exists(os.path.join(checkpoint_path, "config.json")):
    model_path = checkpoint_path
elif os.path.exists(os.path.join(base_path, "config.json")):
    model_path = base_path
else:
    raise FileNotFoundError(f"Could not find model config.json in {base_path} or its checkpoint folder. Please ensure Google Drive is mounted and the path is correct.")

print(f"Loading model from: {model_path}")

# Load the fine-tuned weights from your checkpoint
model = AutoModelForImageClassification.from_pretrained(model_path)

# Load the processor from the original base model, as image transforms didn't change
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
model.eval()

def predict(image):
    inputs = processor(images=image.convert("RGB"), return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    return "Pneumonia" if pred == 1 else "Normal"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Pneumonia X-Ray Classifier (Fine-Tuned ResNet-18)",
)

demo.launch(share=True)

Loading model from: /content/drive/MyDrive/my_ml_models/fine_tuned_pneumonia_model_best_hp_ensemble/model_seed_42/checkpoint-150


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf432a52844d59f481.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
